# 02 — Feature Engineering (Scenarios A / B / C)

Builds the three feature sets used by every model in `03_models.ipynb`


In [1]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import numpy as np
import pandas as pd
from sklearn.preprocessing import PolynomialFeatures

from src.preprocessing import (
    load_raw, split_inputs_target_modes, make_train_test_split,
    NUMERIC_COLS, CATEGORICAL_COLS, FAILURE_MODE_COLS, TARGET_COL,
)
from src.features import PhysicsFeatures, numeric_cols_for_scenario_c

pd.set_option("display.max_columns", 30)
print("Repo root:", REPO_ROOT)


Repo root: /home/melontree/machine-failure-supervised-ml-analysis


## 1. Load data and drop leakage columns

In [2]:
DATA_PATH = REPO_ROOT / "data" / "ai4i2020.csv"
df = load_raw(DATA_PATH)
print("Shape:", df.shape)
df.head(3)


Shape: (10000, 14)


,UDI,Product ID,Type,Air temperature [K],Process temperature [K],Rotational speed [rpm],Torque [Nm],Tool wear [min],Machine failure,TWF,HDF,PWF,OSF,RNF
0,1,M14860,M,298.1,308.6,1551,42.8,0,0,0,0,0,0,0
1,2,L47181,L,298.2,308.7,1408,46.3,3,0,0,0,0,0,0
2,3,L47182,L,298.1,308.5,1498,49.4,5,0,0,0,0,0,0


In [3]:
X, y, modes = split_inputs_target_modes(df)

print("X columns (model inputs only):", list(X.columns))
print("y (target):", TARGET_COL, "| failure rate: %.4f" % y.mean())
print("modes (analysis only, NEVER fed to a model):", list(modes.columns))
assert not set(FAILURE_MODE_COLS) & set(X.columns), "Leakage columns must not be in X"


X columns (model inputs only): ['Air temperature [K]', 'Process temperature [K]', 'Rotational speed [rpm]', 'Torque [Nm]', 'Tool wear [min]', 'Type']
y (target): Machine failure | failure rate: 0.0339
modes (analysis only, NEVER fed to a model): ['TWF', 'HDF', 'PWF', 'OSF', 'RNF']


## 2. Train/test split (stratified, shared by every scenario)

`random_state=42` and `test_size=0.2` are fixed here so every model in `03_models.ipynb` is compared on the exact same rows. **Nothing downstream is fit on the test set.**

In [4]:
X_train, X_test, y_train, y_test, modes_train, modes_test = make_train_test_split(
    X, y, modes, test_size=0.2, random_state=42
)

print("Train:", X_train.shape, "| failure rate: %.4f" % y_train.mean())
print("Test :", X_test.shape,  "| failure rate: %.4f" % y_test.mean())


Train: (8000, 6) | failure rate: 0.0339
Test : (2000, 6) | failure rate: 0.0340


## 3. Scenario A — Raw features

No transformation yet beyond what `ColumnTransformer` will do inside each model's pipeline in `03_models.ipynb` (one-hot `Type`, and scaling for Logistic Regression / KNN only). Shown here as-is.

In [5]:
X_train_A = X_train.copy()
X_test_A  = X_test.copy()

print("Scenario A columns:", list(X_train_A.columns))
X_train_A.head(3)


Scenario A columns: ['Air temperature [K]', 'Process temperature [K]', 'Rotational speed [rpm]', 'Torque [Nm]', 'Tool wear [min]', 'Type']


,Air temperature [K],Process temperature [K],Rotational speed [rpm],Torque [Nm],Tool wear [min],Type
4058,302.0,310.9,1456,47.2,54,M
1221,297.0,308.3,1399,46.4,132,M
6895,301.0,311.6,1357,45.6,137,M


## 4. Scenario B — Generic automatic features

`PolynomialFeatures(degree=2, include_bias=False)` on the five numeric columns only (not on the one-hot `Type` columns, to avoid an explosion of meaningless dummy-squared terms). **Fit on `X_train` only**, then applied to both splits.

In [6]:
poly = PolynomialFeatures(degree=2, include_bias=False)
poly.fit(X_train[NUMERIC_COLS])

poly_train = pd.DataFrame(
    poly.transform(X_train[NUMERIC_COLS]),
    columns=poly.get_feature_names_out(NUMERIC_COLS),
    index=X_train.index,
)
poly_test = pd.DataFrame(
    poly.transform(X_test[NUMERIC_COLS]),
    columns=poly.get_feature_names_out(NUMERIC_COLS),
    index=X_test.index,
)

# Drop the degree-1 columns from poly_* (already in X_train/X_test) to avoid duplicate columns
degree1 = NUMERIC_COLS
poly_only_train = poly_train.drop(columns=degree1)
poly_only_test  = poly_test.drop(columns=degree1)

X_train_B = pd.concat([X_train.copy(), poly_only_train], axis=1)
X_test_B  = pd.concat([X_test.copy(),  poly_only_test],  axis=1)

print("Scenario B shape (train):", X_train_B.shape, "| added", poly_only_train.shape[1], "polynomial columns")
X_train_B.head(3)


Scenario B shape (train): (8000, 21) | added 15 polynomial columns


,Air temperature [K],Process temperature [K],Rotational speed [rpm],Torque [Nm],Tool wear [min],Type,Air temperature [K]^2,Air temperature [K] Process temperature [K],Air temperature [K] Rotational speed [rpm],Air temperature [K] Torque [Nm],Air temperature [K] Tool wear [min],Process temperature [K]^2,Process temperature [K] Rotational speed [rpm],Process temperature [K] Torque [Nm],Process temperature [K] Tool wear [min],Rotational speed [rpm]^2,Rotational speed [rpm] Torque [Nm],Rotational speed [rpm] Tool wear [min],Torque [Nm]^2,Torque [Nm] Tool wear [min],Tool wear [min]^2
4058,302.0,310.9,1456,47.2,54,M,91204.0,93891.8,439712.0,14254.4,16308.0,96658.81,452670.4,14674.48,16788.6,2119936.0,68723.2,78624.0,2227.84,2548.8,2916.0
1221,297.0,308.3,1399,46.4,132,M,88209.0,91565.1,415503.0,13780.8,39204.0,95048.89,431311.7,14305.12,40695.6,1957201.0,64913.6,184668.0,2152.96,6124.8,17424.0
6895,301.0,311.6,1357,45.6,137,M,90601.0,93791.6,408457.0,13725.6,41237.0,97094.56,422841.2,14208.96,42689.2,1841449.0,61879.2,185909.0,2079.36,6247.2,18769.0


## 5. Scenario C — Physics-informed features

`PhysicsFeatures` (from `src/features.py`) adds:
- `temp_diff` = Process temperature − Air temperature (compare with the HDF threshold, 8.6 K)
- `power_W` = Torque × Rotational speed × 2π/60 (compare with the PWF thresholds, 3500 W / 9000 W)
- `wear_x_torque` = Tool wear × Torque (compare with the OSF thresholds, 11000/12000/13000 minNm)
- `osf_margin` = `wear_x_torque` − the Type-specific OSF limit (negative = under the documented limit)

This transformer has **no `fit` state** (it's a pure formula), so fitting on train vs. test makes no difference — but we still call `fit_transform` on train and `transform` on test to keep the pattern consistent with scenario B and with scikit-learn Pipelines.

In [7]:
physics = PhysicsFeatures(add_osf_margin=True)

X_train_C = physics.fit_transform(X_train)
X_test_C  = physics.transform(X_test)

print("Scenario C columns:", list(X_train_C.columns))
X_train_C[["temp_diff", "power_W", "wear_x_torque", "osf_margin"]].describe().T


Scenario C columns: ['Air temperature [K]', 'Process temperature [K]', 'Rotational speed [rpm]', 'Torque [Nm]', 'Tool wear [min]', 'Type', 'temp_diff', 'power_W', 'wear_x_torque', 'osf_margin']


,count,mean,std,min,25%,50%,75%,max
temp_diff,8000.0,10.000613,0.999854,7.60000,9.300000,9.800000,11.000000,12.100000
power_W,8000.0,6282.617947,1072.418510,1148.44061,5559.605833,6272.294453,7013.867399,10469.923005
wear_x_torque,8000.0,4302.684150,2820.177264,0.00000,1953.325000,3998.700000,6265.050000,16497.000000
osf_margin,8000.0,-7192.190850,2909.985619,-13000.00000,-9535.775000,-7463.000000,-5171.825000,4497.000000


### Sanity check: engineered features vs. the documented rules

A quick check that `power_W` and `temp_diff` land where the UCI documentation says a failure of that type should occur, using the training split's own failure-mode flags (analysis only — `modes_train` is never used as a model input).

In [8]:
hdf_hits = modes_train["HDF"] == 1
pwf_hits = modes_train["PWF"] == 1
osf_hits = modes_train["OSF"] == 1

print("HDF rows -> temp_diff < 8.6 and rpm < 1380:",
      float(((X_train_C.loc[hdf_hits, "temp_diff"] < 8.6) &
             (X_train.loc[hdf_hits, "Rotational speed [rpm]"] < 1380)).mean()))

print("PWF rows -> power_W outside [3500, 9000]:",
      float(((X_train_C.loc[pwf_hits, "power_W"] < 3500) |
             (X_train_C.loc[pwf_hits, "power_W"] > 9000)).mean()))

print("OSF rows -> osf_margin > 0:",
      float((X_train_C.loc[osf_hits, "osf_margin"] > 0).mean()))


HDF rows -> temp_diff < 8.6 and rpm < 1380: 1.0
PWF rows -> power_W outside [3500, 9000]: 1.0
OSF rows -> osf_margin > 0: 1.0


## 6. Save the splits and feature sets for `03_models.ipynb`

Everything three notebooks downstream need: the raw split (`X_train`/`X_test`/`y_train`/`y_test`), the failure-mode flags for analysis, and the three scenario feature tables. Saved with `joblib` so dtypes and the DataFrame structure are preserved exactly (unlike CSV round-tripping).

**Note:** this file is an intermediate artifact for the pipeline, not one of the mandatory deliverables in the rulebook — it can be regenerated at any time by re-running this notebook.

In [9]:
import joblib

RESULTS_DIR = REPO_ROOT / "results"
RESULTS_DIR.mkdir(exist_ok=True)

joblib.dump(
    {
        "y_train": y_train, "y_test": y_test,
        "modes_train": modes_train, "modes_test": modes_test,
        "X_train_A": X_train_A, "X_test_A": X_test_A,
        "X_train_B": X_train_B, "X_test_B": X_test_B,
        "X_train_C": X_train_C, "X_test_C": X_test_C,
        "poly_transformer": poly,
    },
    RESULTS_DIR / "feature_splits.joblib",
)

print("Saved:", RESULTS_DIR / "feature_splits.joblib")


Saved: /home/melontree/machine-failure-supervised-ml-analysis/results/feature_splits.joblib


## Summary

| Scenario | Columns (train) | Notes |
|---|---|---|
| A — Raw | see section 3 | 5 numeric + `Type` |
| B — Generic automatic | see section 4 | A + degree-2 polynomial terms |
| C — Physics-informed | see section 5 | A + `temp_diff`, `power_W`, `wear_x_torque`, `osf_margin` |

Next notebook: `03_models.ipynb` loads `results/feature_splits.joblib`, wraps each scenario in a `ColumnTransformer` (scaling for LR/KNN, none for the Decision Tree — see `src/preprocessing.py`), and trains/tunes Logistic Regression, Decision Tree, and KNN on all three scenarios, evaluated on the **test set only**.